# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2: Refresh / Content Opportunity Scoring.**

I'm scoring which pages to review first for refresh, expansion, protection, pruning, or monitoring. I picked this over Lane 4 (CTR/engagement scoring) because the starter dataset already gives me the pieces I need for a *staleness + demand + trend* story — `freshness_tier`, `days_since_last_update`, `search_volume`, and last/prev-30-day clicks — without needing the warehouse's daily table to build a first honest baseline. Lane 4 is narrower (title/meta/snippet review), and I can fold a CTR-gap check into Lane 2's reason codes later if it turns out to matter. I'm keeping this provisional until Week 4, when I'll know if a proper future-window label (built from the warehouse's daily table, not the `trend_direction` proxy) actually separates recoverable pages from dead ones.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Out of a client's full content inventory, which pages should an editor spend this sprint's limited refresh hours on first?

**Who acts, and how:** A content editor or SEO analyst pulls the top N rows from my ranked queue at the start of a sprint and works down the list instead of picking pages by gut feel or "whatever's on my mind."

**Cost of a wrong call:**
- *False positive* (I flag a page as worth refreshing, it wasn't): wasted editor hours — usually 2-6 hours per refresh — spent on a page that wouldn't have recovered regardless. The real cost isn't the hours alone, it's the opportunity cost: those hours weren't spent on a page that *would* have recovered.
- *False negative* (a genuinely declining, high-demand page never surfaces): the client keeps losing visibility and clicks on a page worth saving, and nobody notices until the drop is much larger and harder to reverse.

Given that, my ranking has to be precision-conscious at the top (the pages an editor actually sees this sprint) more than exhaustive at the bottom.

**Why a plain rule isn't enough:** a single if-statement like "flag anything stale" catches too much — 31% of the dataset is stale by freshness tier alone, and most of that isn't declining. I need to combine staleness, real search demand, and an observed trend signal, and weigh them against each other — that's a scoring/ranking problem, not a single threshold.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd
from pathlib import Path

# works whether the notebook runs from repo root or from work/notebooks/
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
csv_path = next(p for p in candidates if p.exists())
df = pd.read_csv(csv_path)

print(f"{len(df):,} rows, {df['client_id'].nunique()} clients")

# 1) how much of the inventory is stale
stale = df[df["freshness_tier"].isin(["91-180", "181+"])]
print(f"Stale content (freshness_tier 91-180 or 181+): {len(stale):,} rows "
      f"({len(stale) / len(df) * 100:.1f}% of all content)")

# 2) of the stale content, how much is ALSO observed declining
stale_declining = stale[stale["trend_direction"] == "down"]
print(f"Stale AND declining (trend_direction == 'down'): {len(stale_declining):,} rows "
      f"({len(stale_declining) / len(stale) * 100:.1f}% of stale content)")

# 3) does 'declining' correspond to a real, measurable click drop?
for label in ["down", "stable", "up"]:
    grp = df[df["trend_direction"] == label]
    last30 = grp["clicks_last_30d"].mean()
    prev30 = grp["clicks_prev_30d"].mean()
    print(f"  trend_direction={label:7s} n={len(grp):5,d}  "
          f"avg clicks_last_30d={last30:5.2f}  avg clicks_prev_30d={prev30:5.2f}  "
          f"delta={last30 - prev30:+.2f}")


30,000 rows, 32 clients
Stale content (freshness_tier 91-180 or 181+): 9,345 rows (31.1% of all content)
Stale AND declining (trend_direction == 'down'): 5,686 rows (60.8% of stale content)
  trend_direction=down    n=16,262  avg clicks_last_30d= 3.35  avg clicks_prev_30d= 4.69  delta=-1.33
  trend_direction=stable  n=5,962  avg clicks_last_30d=11.08  avg clicks_prev_30d=11.55  delta=-0.48
  trend_direction=up      n=4,388  avg clicks_last_30d= 6.09  avg clicks_prev_30d= 4.09  delta=+1.99


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- *Observed*: "9,345 of 30,000 pages (31.1%) are stale by freshness tier; 5,686 of those (60.8%) are also observed declining." These are counts from the data, not predictions.
- *Directional*: pages tagged `trend_direction == 'down'` show a measured drop in average clicks between the previous and last 30-day windows (4.69 → 3.35, about -28%), while `stable` pages are roughly flat and `up` pages gain. That's an association I measured, not a mechanism I proved.
- *Decision-support*: a ranked queue with reason codes is a prioritization tool for a human editor — it surfaces candidates, it doesn't make the call.

**What I will never claim:**
- That refreshing a flagged page *causes* recovery — I have no experiment (no A/B test, no before/after on refreshed vs. matched unrefreshed pages), only observational data.
- That anything here predicts or explains a Google ranking factor — I have client-side, pseudonymized metrics, not Google's algorithm.
- Any claim built on `trend_direction` or `trend_pct` as a model *feature* — both are derived from the same signal I'd be trying to predict, so using them as inputs would be leakage, not a real prediction. I'll only use them here for *descriptive* stats (as above) — by Weeks 4-5 the real target needs to come from a future time window in the warehouse's daily table, observed after the fact, not computed from a rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.